# Clase 12 — Colores en la terminal

---

## ¿Cómo funciona?

La terminal interpreta secuencias especiales de caracteres llamadas **códigos de escape ANSI**. Son invisibles para el usuario pero cambian el comportamiento del texto que viene después.

Toda secuencia tiene la misma estructura:

```
\033[  <código>  m
  │       │      │
  │       │      └─ cierra la secuencia
  │       └─ número que indica qué cambiar
  └─ inicio de secuencia (ESC + corchete)
```

`\033` es el carácter ESC en octal. También se puede escribir `\x1b` (hexadecimal) o `\e` — las tres formas son equivalentes.

In [ ]:
# Las tres formas son idénticas
print("\033[31m rojo con octal  \033[0m")
print("\x1b[31m rojo con hex    \x1b[0m")

# Sin el reset al final, el color se queda
print("\033[31m esto es rojo...")
print("...esto también")
print("\033[0m ya no")

---

## Colores de texto (foreground)

In [ ]:
colores_texto = {
    "negro"  : 30,
    "rojo"   : 31,
    "verde"  : 32,
    "amarillo": 33,
    "azul"   : 34,
    "magenta": 35,
    "cyan"   : 36,
    "blanco" : 37,
}

for nombre, codigo in colores_texto.items():
    print(f"  \033[{codigo}m {nombre:10} \033[0m  código: {codigo}")

In [ ]:
# Variantes brillantes: sumar 60 al código
colores_brillantes = {
    "negro brillante"  : 90,
    "rojo brillante"   : 91,
    "verde brillante"  : 92,
    "amarillo brillante": 93,
    "azul brillante"   : 94,
    "magenta brillante": 95,
    "cyan brillante"   : 96,
    "blanco brillante" : 97,
}

for nombre, codigo in colores_brillantes.items():
    print(f"  \033[{codigo}m {nombre:22} \033[0m  código: {codigo}")

---

## Colores de fondo (background)

Misma lógica, pero los códigos empiezan en 40 (normales) y 100 (brillantes).

In [ ]:
fondos = {
    "negro"  : 40,
    "rojo"   : 41,
    "verde"  : 42,
    "amarillo": 43,
    "azul"   : 44,
    "magenta": 45,
    "cyan"   : 46,
    "blanco" : 47,
}

for nombre, codigo in fondos.items():
    print(f"  \033[{codigo}m  {nombre:10}  \033[0m  código: {codigo}")

---

## Estilos de texto

In [ ]:
estilos = {
    "negrita"      : 1,
    "tenue"        : 2,
    "cursiva"      : 3,
    "subrayado"    : 4,
    "invertido"    : 7,
    "tachado"      : 9,
}

for nombre, codigo in estilos.items():
    print(f"  \033[{codigo}m {nombre:15} \033[0m  código: {codigo}")

---

## Combinar estilos

Los códigos se separan con `;` dentro de la misma secuencia.

In [ ]:
# texto + fondo
print("\033[33;44m  amarillo sobre azul  \033[0m")

# texto + estilo
print("\033[1;32m  verde en negrita      \033[0m")

# texto + fondo + estilo
print("\033[1;4;97;41m  blanco negrita subrayado sobre rojo  \033[0m")

# se pueden combinar todos los que quieras
print("\033[3;35m  magenta en cursiva    \033[0m")

---

## 256 colores

Los 8 colores básicos son solo el principio. Con una sintaxis extendida se puede elegir entre **256 colores**:

```
\033[38;5;<n>m   →  color de texto número n
\033[48;5;<n>m   →  color de fondo número n
```

Los primeros 16 son los colores básicos ya vistos. Del 16 al 231 hay un cubo de colores. Del 232 al 255 hay una escala de grises.

In [ ]:
# Paleta completa de los 256 colores de fondo
print("\nPaleta de 256 colores:\n")

for n in range(256):
    print(f"\033[48;5;{n}m  {n:3}  \033[0m", end="")
    if (n + 1) % 16 == 0:
        print()

---

## Color RGB (true color)

La mayoría de terminales modernas soportan los 16 millones de colores RGB:

```
\033[38;2;<R>;<G>;<B>m   →  texto con color RGB
\033[48;2;<R>;<G>;<B>m   →  fondo con color RGB
```

In [ ]:
# Texto con colores RGB exactos
print("\033[38;2;255;100;0m  naranja exacto   \033[0m")
print("\033[38;2;138;43;226m  violeta exacto   \033[0m")
print("\033[48;2;30;30;30;m\033[38;2;0;255;180m  fondo oscuro, texto turquesa  \033[0m")

# Degradado horizontal de rojo a azul
print("\nDegradado rojo → azul:")
for i in range(40):
    r = int(255 * (1 - i / 39))
    b = int(255 * (i / 39))
    print(f"\033[48;2;{r};0;{b}m  \033[0m", end="")
print()

---

## Una clase `Color` para no recordar los números

En vez de escribir `\033[32m` cada vez, podemos encapsular todo en una clase.

In [ ]:
class Color:
    _RESET = "\033[0m"

    @staticmethod
    def texto(r, g, b):
        return f"\033[38;2;{r};{g};{b}m"

    @staticmethod
    def fondo(r, g, b):
        return f"\033[48;2;{r};{g};{b}m"

    @staticmethod
    def reset():
        return Color._RESET

    @staticmethod
    def pintar(texto, r_texto, g_texto, b_texto, r_fondo=None, g_fondo=None, b_fondo=None):
        inicio = Color.texto(r_texto, g_texto, b_texto)
        if r_fondo is not None:
            inicio += Color.fondo(r_fondo, g_fondo, b_fondo)
        return f"{inicio}{texto}{Color._RESET}"


# Uso
print(Color.pintar("  Hola mundo  ", 255, 255, 255, 70, 130, 180))
print(Color.pintar("  Error crítico  ", 255, 255, 255, 180, 0, 0))
print(Color.pintar("  Todo OK  ", 0, 0, 0, 50, 200, 50))

---

## Demo: barra de progreso

In [ ]:
import time

def barra_progreso(actual, total, ancho=40):
    porcentaje = actual / total
    relleno    = int(ancho * porcentaje)
    vacio      = ancho - relleno

    # color varía de rojo a verde según el avance
    r = int(255 * (1 - porcentaje))
    g = int(255 * porcentaje)

    barra = f"\033[48;2;{r};{g};0m{' ' * relleno}\033[0m{' ' * vacio}"
    print(f"\r  [{barra}] {actual}/{total} ({porcentaje:.0%}) ", end="", flush=True)

total = 30
for i in range(total + 1):
    barra_progreso(i, total)
    time.sleep(0.05)
print()

---

## Ejercicios

---

### Ejercicio 1

Imprime una tabla de multiplicar del 1 al 10 donde:
- Los encabezados de fila y columna van en **negrita**
- Los múltiplos de 5 tienen **fondo amarillo**
- Los múltiplos de 10 tienen **fondo verde**

In [ ]:
# Ejercicio 1


---

### Ejercicio 2

Crea una función `imprimir_log(nivel, mensaje)` donde `nivel` puede ser `"info"`, `"aviso"` o `"error"`, y cada uno imprime con un color distinto y un prefijo:

```
[ INFO  ]  Servidor iniciado
[ AVISO ]  Memoria al 80%
[ ERROR ]  Conexión rechazada
```

In [ ]:
# Ejercicio 2


---

### Ejercicio 3

Usando RGB, imprime el texto `"ARCOÍRIS"` con cada letra en un color distinto del espectro visible (rojo, naranja, amarillo, verde, azul, índigo, violeta).

In [ ]:
# Ejercicio 3
